In [ ]:
# Copy each "# %%" block into its own Kaggle cell. Run them ONE AT A TIME, in order.
# Dataset: "Chest X-Ray Images (Pneumonia)" (Kermany), already added as an input.
# Output goes to /kaggle/working/clean_kermany. Old models and folders are kept.

# %% ─── CELL 1: SETUP + SHARED PREPROCESSING ─────────────────────────────────
import os, re, glob, time, random, hashlib, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupShuffleSplit

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# find the dataset folder automatically
cands = []
for pat in ["/kaggle/input/*/chest_xray", "/kaggle/input/*/*/chest_xray", "/kaggle/input/*/*/*/chest_xray"]:
    cands += [p for p in glob.glob(pat) if os.path.isdir(os.path.join(p, "train"))]
assert cands, "chest_xray folder not found: check that the Kermany dataset is added as an input"
ROOT = cands[0]
print("Dataset root:", ROOT)

CLEAN    = "/kaggle/working/clean_kermany"
CLASSES  = ["NORMAL", "PNEUMONIA"]          # folder names in Kermany
IMG_SIZE = (256, 256)
CROP     = 0.05

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_gray(g):
    """THE preprocessing. Training, evaluation, external tests and Flask must all use this."""
    h, w = g.shape
    dy, dx = int(h * CROP), int(w * CROP)
    g = g[dy:h - dy, dx:w - dx]
    g = cv2.resize(g, IMG_SIZE, interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(g, (1, 99))
    g = np.clip((g.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
    g = (g * 255).astype(np.uint8)
    return clahe.apply(g)

def preprocess_file(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return None if g is None else preprocess_gray(g)


# %% ─── CELL 2: BUILD THE CLEAN DATASET (split BY PATIENT) ───────────────────
# Kermany filenames contain a patient id: person12_bacteria_45.jpeg (pneumonia),
# IM-0115-0001.jpeg / NORMAL2-IM-0373-0001.jpeg (normal).
# Images of the same patient are never split between train and val.

def patient_id(cls, fname):
    m = re.match(r"person(\d+)_", fname) if cls == "PNEUMONIA" else re.search(r"IM-(\d+)", fname)
    return cls + (m.group(1) if m else fname)          # fallback: the image is its own group

def list_files(split):
    out = []
    for c in CLASSES:
        d = os.path.join(ROOT, split, c)
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                out.append((c, f, os.path.join(d, f)))
    return out

def work(t):
    c, f, p = t
    a = preprocess_file(p)
    return None if a is None else (c, f, hashlib.md5(a.tobytes()).hexdigest(), a)

def collect(split):
    files, t0 = list_files(split), time.time()
    with ThreadPoolExecutor(8) as ex:
        res = [r for r in ex.map(work, files) if r is not None]
    print(f"{split}: {len(res)} images read in {time.time()-t0:.0f}s", flush=True)
    return res

pool = collect("train") + collect("val")      # val is tiny (16 images): merge and re-split
test = collect("test")                         # original test set: kept as is

# exact duplicates
seen, pool_u = set(), []
for c, f, h, a in pool:
    if h not in seen:
        seen.add(h); pool_u.append((c, f, h, a))
test_u = [t for t in test if t[2] not in seen]
print(f"pool: {len(pool)} -> {len(pool_u)} | test: {len(test)} -> {len(test_u)}")

# patient-level split of the pool
y_pool = np.array([CLASSES.index(c) for c, _, _, _ in pool_u])
groups = [patient_id(c, f) for c, f, _, _ in pool_u]
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
                      .split(np.zeros(len(pool_u)), y_pool, groups))
assert not ({groups[i] for i in tr_idx} & {groups[i] for i in va_idx}), "patient leakage!"
print("patients in train:", len({groups[i] for i in tr_idx}), "| in val:", len({groups[i] for i in va_idx}))

def write(items, split):
    for c in CLASSES:
        os.makedirs(os.path.join(CLEAN, split, c), exist_ok=True)
    counters = {c: 0 for c in CLASSES}
    random.shuffle(items)
    for c, _, _, a in items:
        cv2.imwrite(os.path.join(CLEAN, split, c, f"{c}_{counters[c]}.png"), a)   # neutral names
        counters[c] += 1
    print(split, counters)

shutil.rmtree(CLEAN, ignore_errors=True)
write([pool_u[i] for i in tr_idx], "train")
write([pool_u[i] for i in va_idx], "val")
write(test_u, "test")
del pool, pool_u, test, test_u


# Run AFTER cells 1-2 of train_kermany.py (the clean dataset must exist in CLEAN).
# Needs: Notebook settings -> Internet ON (to download the ImageNet weights), GPU ON.
# Replaces cells 3 and 5. Then run cell 6 (evaluation) and the threshold cell as before.

# %% ─── CELL A: RGB GENERATORS (the pretrained backbone expects 3 channels) ──
from tensorflow.keras.applications import DenseNet121

BATCH_SIZE = 32

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=7, zoom_range=0.1,
                                   width_shift_range=0.05, height_shift_range=0.05,
                                   fill_mode="constant", cval=0)
eval_datagen  = ImageDataGenerator(rescale=1./255)

def flow_rgb(datagen, split, shuffle):
    return datagen.flow_from_directory(os.path.join(CLEAN, split), target_size=IMG_SIZE,
                                       color_mode="rgb", batch_size=BATCH_SIZE,
                                       classes=CLASSES, class_mode="binary", shuffle=shuffle)

train_gen = flow_rgb(train_datagen, "train", True)
val_gen   = flow_rgb(eval_datagen,  "val",   False)
test_gen  = flow_rgb(eval_datagen,  "test",  False)

cw = compute_class_weight("balanced", classes=np.unique(train_gen.classes), y=train_gen.classes)
class_weights = dict(enumerate(cw))
print("Class weights:", class_weights)


# %% ─── CELL B: MODEL (DenseNet121) + 2-PHASE TRAINING ──────────────────────
base = DenseNet121(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base.trainable = False

inp = layers.Input((*IMG_SIZE, 3))
x = layers.Normalization(axis=-1, mean=[0.485, 0.456, 0.406],
                         variance=[0.229**2, 0.224**2, 0.225**2])(inp)   # ImageNet statistics
x = base(x, training=False)                 # BatchNorm stays in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation="sigmoid")(x)
model = models.Model(inp, out)

MODEL_PATH = "/kaggle/working/densenet_kermany.keras"

# Phase 1: train only the new head
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
h1 = model.fit(train_gen, validation_data=val_gen, epochs=6, class_weight=class_weights)

# Phase 2: fine-tune the last layers of the backbone with a very small learning rate
base.trainable = True
for l in base.layers[:-60]:
    l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization):
        l.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="binary_crossentropy", metrics=["accuracy"])
cbs = [
    callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]
history = model.fit(train_gen, validation_data=val_gen, epochs=25,
                    callbacks=cbs, class_weight=class_weights)

# Now run CELL 6 (evaluation) and the threshold cell unchanged.
# Note: the Grad-CAM and MC-Dropout cells need adapting for this model (see message).

# %% ─── CELL 6: EVALUATION ───────────────────────────────────────────────────
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, auc)
SAVE_DIR = "/kaggle/working/evaluation_kermany"; os.makedirs(SAVE_DIR, exist_ok=True)

y_true = test_gen.classes
probs  = model.predict(test_gen, verbose=0).ravel()
y_pred = (probs > 0.5).astype(int)

report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
print(report); open(f"{SAVE_DIR}/classification_report.txt", "w").write(report)

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(cmap="Blues")
plt.title("Confusion matrix (Kermany, patient split)"); plt.savefig(f"{SAVE_DIR}/confusion_matrix.png"); plt.show()

fpr, tpr, _ = roc_curve(y_true, probs); roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5)); plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}"); plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(); plt.grid(True)
plt.savefig(f"{SAVE_DIR}/roc_curve.png"); plt.show()

for k in ["accuracy", "val_accuracy", "loss", "val_loss"]:
    plt.plot(history.history[k], label=k)
plt.legend(); plt.grid(True); plt.savefig(f"{SAVE_DIR}/training_history.png"); plt.show()

from sklearn.metrics import roc_curve
pv = model.predict(val_gen, verbose=0).ravel(); yv = val_gen.classes
fpr, tpr, thr = roc_curve(yv, pv)
t_best = thr[np.argmax(tpr - fpr)]
print("threshold:", round(float(t_best), 3))
yp = (probs > t_best).astype(int)
print(classification_report(y_true, yp, target_names=CLASSES, digits=4))

# Run right after CELL 6 (needs: probs, y_true, CLASSES, SEED, classification_report).
# probs = model.predict(test_gen).ravel() on the DenseNet model.
from sklearn.metrics import roc_auc_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

print("AUC (Kermany test):", round(roc_auc_score(y_true, probs), 4))

# ── 1) Trade-off table: ANALYSIS ONLY (do not pick the headline threshold from the full test set)
print("\nthr | Normal recall | Pneumonia recall | Normal F1 | Pneumonia F1 | accuracy")
for t in np.arange(0.30, 0.96, 0.05):
    yp = (probs > t).astype(int)
    print(f"{t:.2f} |     {recall_score(y_true, yp, pos_label=0):.3f}     |      "
          f"{recall_score(y_true, yp, pos_label=1):.3f}       |   "
          f"{f1_score(y_true, yp, pos_label=0):.3f}   |    {f1_score(y_true, yp, pos_label=1):.3f}     | "
          f"{(yp == y_true).mean():.3f}")

# ── 2) Honest selection: pick the threshold on one half of the test set, report on the other half
idx_a, idx_b = train_test_split(np.arange(len(y_true)), test_size=0.5,
                                stratify=y_true, random_state=SEED)

def pick_threshold(idx):
    """Threshold that maximizes the WORST of the two recalls (balances Normal and Pneumonia)."""
    best_t, best_s = 0.5, -1
    for t in np.arange(0.20, 0.96, 0.01):
        yp = (probs[idx] > t).astype(int)
        s = min(recall_score(y_true[idx], yp, pos_label=0),
                recall_score(y_true[idx], yp, pos_label=1))
        if s > best_s:
            best_t, best_s = t, s
    return best_t

for name, (fit, ev) in {"A -> B": (idx_a, idx_b), "B -> A": (idx_b, idx_a)}.items():
    t = pick_threshold(fit)
    yp = (probs[ev] > t).astype(int)
    print(f"\n=== threshold chosen on half {name[0]}: {t:.2f}  | reported on half {name[-1]} ===")
    print(classification_report(y_true[ev], yp, target_names=CLASSES, digits=3))


# %% ─── CELL 7: EXTERNAL TEST (NIH sample, adult images) ─────────────────────
# Add the input "NIH Chest X-rays" (sample version) first. File names may differ: adapt if needed.
import pandas as pd
from sklearn.metrics import classification_report as cr

EXT = "/kaggle/working/external"
csvs = glob.glob("/kaggle/input/**/sample_labels.csv", recursive=True)
if csvs and not os.path.isdir(f"{EXT}/Normal"):
    df = pd.read_csv(csvs[0])
    img_paths = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/*.png", recursive=True)}
    sel = {"Normal":    df[df["Finding Labels"] == "No Finding"]["Image Index"].tolist()[:40],
           "Pneumonia": df[df["Finding Labels"].str.contains("Pneumonia")]["Image Index"].tolist()[:40]}
    for cls, names in sel.items():
        os.makedirs(f"{EXT}/{cls}", exist_ok=True)
        for n in names:
            if n in img_paths: shutil.copy(img_paths[n], f"{EXT}/{cls}/{n}")

X_ext, y_ext, names = [], [], []
for ci, cname in enumerate(["Normal", "Pneumonia"]):
    d = f"{EXT}/{cname}"
    if not os.path.isdir(d): continue
    for f in sorted(os.listdir(d)):
        a = preprocess_file(os.path.join(d, f))
        if a is not None:
            X_ext.append(np.repeat(a.astype("float32")[..., None] / 255.0, 3, axis=-1)); y_ext.append(ci); names.append(f"{cname}/{f}")

if not X_ext:
    print("No external images found.")
else:
    X_ext, y_ext = np.stack(X_ext), np.array(y_ext)
    p_ext = model.predict(X_ext, verbose=0).ravel()
    print(cr(y_ext, (p_ext > 0.5).astype(int), labels=[0, 1],
             target_names=["Normal", "Pneumonia"], zero_division=0))


# %% ─── CELL 8: FLASK (replace preprocess_image in app.py) ──────────────────
# Copy CROP, IMG_SIZE, clahe and preprocess_gray into app.py, then:
#
# def preprocess_image(pil_image):
#     g = np.asarray(pil_image.convert("L"), dtype=np.uint8)
#     g = preprocess_gray(g)
#     return g.astype(np.float32)[..., None][None, ...] / 255.0     # (1, 256, 256, 1)
#
# MODEL_PATH = "simple_best_binary_kermany.keras"
# The Grad-CAM code in app.py stays the same.

In [ ]:
# Run in the DenseNet notebook, after cell 1 (preprocess_file) and after the model is trained/loaded.
# The old dataset (binary-dataset1) must still be attached as an input.
# We use ONLY images the model cannot have seen:
#   - old Normal images (adult, another collection)
#   - old "VP" pneumonia images (another collection)
# We EXCLUDE "PNMonia" images: they are Kermany images, probably seen in training.

import random
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import classification_report

OLD = "/kaggle/input/datasets/raoufbouk/binary-dataset1/binary_dataset"

def old_files(cls, prefix=None):
    out = []
    for sp in ["train", "val", "test"]:
        d = os.path.join(OLD, sp, cls)
        if os.path.isdir(d):
            out += [os.path.join(d, f) for f in os.listdir(d)
                    if prefix is None or f.startswith(prefix)]
    return out

rng = random.Random(0)
normal = rng.sample(old_files("Normal"), 500)
vp_all = old_files("Pneumonia", "VP")
vp = rng.sample(vp_all, min(500, len(vp_all)))
print(len(normal), "normal |", len(vp), "VP pneumonia")

def load(path):
    a = preprocess_file(path)                     # same preprocessing as training
    if a is None:
        return None
    return np.repeat((a.astype("float32") / 255.0)[..., None], 3, axis=-1)   # 3 channels for DenseNet

paths  = normal + vp
labels = [0] * len(normal) + [1] * len(vp)
with ThreadPoolExecutor(8) as ex:
    arrs = list(ex.map(load, paths))

keep = [i for i, a in enumerate(arrs) if a is not None]
X = np.stack([arrs[i] for i in keep])
y = np.array([labels[i] for i in keep])

p = model.predict(X, batch_size=32, verbose=0).ravel()
print(classification_report(y, (p > 0.5).astype(int),
                            target_names=["Normal (old)", "VP pneumonia (old)"], digits=3))
print("Mean P(pneumonia): normal =", round(float(p[y == 0].mean()), 3),
      "| VP =", round(float(p[y == 1].mean()), 3))

In [ ]:
import numpy as np
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from tensorflow.keras.preprocessing import image

# 1. Model (weights unchanged)
model_cam = tf.keras.models.load_model("/kaggle/working/densenet_kermany.keras")
last_conv = [l.name for l in model_cam.layers if isinstance(l, tf.keras.layers.Conv2D)][-1]
JET = plt.get_cmap("jet")   # do not use "cm": your evaluation cell already uses that name for the confusion matrix

# 2. Chest region (display only, the model never sees this mask)
ROI_CENTER_Y, ROI_RADIUS_Y = 0.52, 0.40
ROI_CENTER_X, ROI_RADIUS_X = 0.50, 0.42
MIN_INSIDE_RATIO = 0.50

def chest_roi(gray, thr=0.06):
    h, w = gray.shape
    rows = np.where(gray.mean(axis=1) > thr)[0]
    cols = np.where(gray.mean(axis=0) > thr)[0]
    if len(rows) < 0.3 * h or len(cols) < 0.3 * w:
        y0, y1, x0, x1 = 0, h, 0, w
    else:
        y0, y1, x0, x1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1
    bh, bw = y1 - y0, x1 - x0
    cy, cx = y0 + ROI_CENTER_Y * bh, x0 + ROI_CENTER_X * bw
    ry, rx = ROI_RADIUS_Y * bh, ROI_RADIUS_X * bw
    yy, xx = np.mgrid[0:h, 0:w]
    inside = (((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2) <= 1.0
    soft = Image.fromarray((inside * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(6))
    return np.asarray(soft, dtype=np.float32) / 255.0

# 3. Grad-CAM on the logit of the predicted class
def gradcam(model, x, last_conv, class_idx):
    with tf.GradientTape() as tape:
        h, conv = x, None
        for layer in model.layers[:-1]:
            h = layer(h, training=False)
            if layer.name == last_conv:
                conv = h
        last = model.layers[-1]
        logit = tf.matmul(h, last.kernel) + last.bias
        score = logit[:, 0] if class_idx == 1 else -logit[:, 0]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv * w[:, None, None, :], axis=-1))[0].numpy()
    return cam

# 4. Explain one image
def explain(img_path, out_path="/kaggle/working/gradcam.png", show_raw=True):
    img = image.load_img(img_path, target_size=(256, 256), color_mode="grayscale")
    x = image.img_to_array(img) / 255.0
    xb = x[None]
    prob = float(model_cam.predict(xb, verbose=0)[0][0])
    idx = int(prob > 0.5)
    label = "Pneumonia" if idx else "Normal"
    conf = prob if idx else 1 - prob

    cam = gradcam(model_cam, tf.convert_to_tensor(xb), last_conv, idx)
    cam_img = Image.fromarray(np.uint8(cam / (cam.max() + 1e-8) * 255))
    cam_img = cam_img.resize((256, 256), Image.BICUBIC).filter(ImageFilter.GaussianBlur(4))
    heat_raw = np.asarray(cam_img, dtype=np.float32) / 255.0

    gray = x.squeeze()
    roi = chest_roi(gray)
    inside_ratio = float((heat_raw * roi).sum() / (heat_raw.sum() + 1e-8))
    suspect = inside_ratio < MIN_INSIDE_RATIO

    def overlay(heat):
        a = (0.65 * heat)[..., None]
        return np.clip(np.stack([gray] * 3, -1) * (1 - a) + JET(heat)[..., :3] * a, 0, 1)

    heat = heat_raw * roi
    heat = heat / (heat.max() + 1e-8)
    heat[heat < 0.30] = 0

    n = 3 if show_raw else 2
    plt.figure(figsize=(5 * n, 5))
    plt.subplot(1, n, 1); plt.imshow(gray, cmap="gray"); plt.axis("off"); plt.title("Original")
    if show_raw:
        plt.subplot(1, n, 2); plt.imshow(overlay(heat_raw / (heat_raw.max() + 1e-8))); plt.axis("off")
        plt.title("Raw Grad-CAM (what the model used)")
    plt.subplot(1, n, n); plt.imshow(overlay(heat)); plt.axis("off")
    plt.title(f"Chest Grad-CAM | {label} {conf*100:.1f}%\ninside_ratio={inside_ratio:.2f}"
              + (" ⚠ suspect" if suspect else ""))
    plt.tight_layout(); plt.savefig(out_path); plt.show()
    return {"label": label, "confidence": conf, "inside_ratio": inside_ratio, "suspect": suspect}

# 5. Test on a random image
idx = np.random.randint(len(test_gen.filenames))
print(explain(os.path.join(test_gen.directory, test_gen.filenames[idx])))